## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-08-13-52-25 +0000,nypost,Small plane hits truck during heart-pounding e...,https://nypost.com/2026/11/08/us-news/small-pl...
1,2026-11-08-13-39-41 +0000,wapo,"Mass shooter kills six in Poland, and three bo...",https://www.washingtonpost.com/world/2026/11/0...
2,2026-11-08-13-30-13 +0000,cbc,Poland in mourning after shooter with 'ultrana...,https://www.cbc.ca/news/world/krakow-shooting-...
3,2026-11-08-13-19-56 +0000,nypost,Russia strikes Ukraine emergency rescue shelte...,https://nypost.com/2026/11/08/world-news/russi...
4,2026-11-08-13-14-09 +0000,nyt,Samoan Chef Brings Flavors of Home to New Zeal...,https://www.nytimes.com/2026/11/08/world/asia/...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2607/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
185,trump,29
94,white,15
43,new,14
313,president,13
100,democrats,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
112,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...,135
168,2026-11-07-18-59-01 +0000,nypost,Trump offers UFC boss Dana White role of White...,https://nypost.com/2026/11/07/us-news/trump-of...,110
116,2026-11-08-00-08-30 +0000,wapo,Trump says he offered White House press secret...,https://www.washingtonpost.com/politics/2026/1...,107
110,2026-11-08-00-31-53 +0000,bbc,"'Dana, please take the job!' - Trump asks UFC ...",https://www.bbc.co.uk/news/articles/cvze47p9je...,93
109,2026-11-08-00-35-42 +0000,cbc,UFC boss Dana White says he's meeting with Tru...,https://www.cbc.ca/news/world/trump-press-secr...,89


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
112,135,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...
160,55,2026-11-07-19-46-03 +0000,cbc,"Israel's president asks Gadi Eisenkot, Netanya...",https://www.cbc.ca/news/world/gadi-eisenkot-fo...
92,47,2026-11-08-02-00-00 +0000,wsj,A dominant midterm performance has given Democ...,https://www.wsj.com/politics/elections/the-new...
142,42,2026-11-07-21-01-09 +0000,nypost,Shocking footage shows moment Krakow police sh...,https://nypost.com/2026/11/07/world-news/shock...
207,42,2026-11-07-16-00-28 +0000,chitri,Mary Miller vs Jennifer Todd: How Illinois’ 15...,https://www.chicagotribune.com/2026/11/07/mary...
172,39,2026-11-07-18-49-11 +0000,nypost,Marco Rubio warns top staff against conflicts ...,https://nypost.com/2026/11/07/us-news/marco-ru...
206,33,2026-11-07-16-09-00 +0000,wsj,Is AI a white-collar job killer? Strong hiring...,https://www.wsj.com/economy/jobs/is-ai-taking-...
120,33,2026-11-07-23-29-51 +0000,nypost,"Staten Island bucks Dems’ blue tsunami, reject...",https://nypost.com/2026/11/07/us-news/staten-i...
154,32,2026-11-07-20-12-26 +0000,nyt,"Father Calls Daughter, the Co-Pilot Killed in ...",https://www.nytimes.com/2026/11/07/us/florida-...
84,31,2026-11-08-04-49-16 +0000,nypost,Ex-cop Sayed Mohassib had NYC jogger Katie Sua...,https://nypost.com/2026/11/07/us-news/ex-cop-s...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
